# Risk Scoring Notebook

This notebook is dedicated to delay risk scoring using probability from a classification model.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
file_path = "/home/pratyush_device/Documents/college/AIML lab/Project/customer_cleaned.csv"
df = pd.read_csv(file_path)
print("Loaded shape:", df.shape)
df.head()

In [ ]:
target_col = "lead_time_days"
if target_col not in df.columns:
    raise ValueError("lead_time_days column not found in customer_cleaned.csv")

# Build delay target from lead_time_days
delay_threshold = df[target_col].quantile(0.75)
df["delay"] = (df[target_col] > delay_threshold).astype(int)

print("Delay threshold:", delay_threshold)
print(df["delay"].value_counts(normalize=True))

In [ ]:
# Prepare model features
X_raw = df.drop(columns=[target_col, "delay"])
X = pd.get_dummies(X_raw, drop_first=True)
y = df["delay"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
def get_risk(prob):
    if prob > 0.7:
        return "High"
    elif prob > 0.4:
        return "Medium"
    return "Low"


def recommend_action(risk):
    if risk == "High":
        return "Expedite shipment or change supplier"
    if risk == "Medium":
        return "Monitor order closely"
    return "No action needed"

In [ ]:
# Risk scoring for all orders
all_prob = clf.predict_proba(X)[:, 1]
scored = X_raw.copy()
scored["Delay_Probability"] = all_prob
scored["Risk_Level"] = scored["Delay_Probability"].apply(get_risk)
scored["Recommended_Action"] = scored["Risk_Level"].apply(recommend_action)

print(scored["Risk_Level"].value_counts())
scored.head(10)

In [ ]:
# Single order risk prediction helper
def predict_order_risk(input_row):
    input_encoded = pd.get_dummies(input_row, drop_first=True)
    input_encoded = input_encoded.reindex(columns=X.columns, fill_value=0)
    prob = clf.predict_proba(input_encoded)[0][1]
    risk = get_risk(prob)
    return {
        "Delay Probability": float(prob),
        "Risk Level": risk,
        "Recommended Action": recommend_action(risk)
    }

sample = X_raw.iloc[[0]]
predict_order_risk(sample)

In [ ]:
output_path = "/home/pratyush_device/Documents/college/AIML lab/Project/risk_scored_orders.csv"
scored.to_csv(output_path, index=False)
print("Saved risk scored file to:", output_path)